# Visualizing fire scars through false color

**Author:** Monique A. Hernandez

**Date:** November 21, 2026

[GitHub Repository](https://github.com/moniqueallyson/eds220-hwk4)


## 1. About Section

### Purpose

The purpose of this notebook is to explore the environmental impact caused by the fires in Palisades and Eaton that occurred almost simultaneously in Los Angeles County at the start of January 2025. The final output for this notebook is a false color image utilizing remote sensing techniques after data exploration and cleaning to visualize the affected areas through the use of the fire perimeters dataset. Data collected by satellites are useful when assigning infrared to visible light to create images that highlight vegetation health, burn severity, and extent of fire scars, which can then help researchers plan restoration strategies and identify high-risk areas for future planning.

### Highlights


- Restoration of CRS through exploratory retrieval using the correct variable, `spatial_ref.crs_wkt`.

- Correct true color imagery by filling in `NaN` values with 0 for bands using RGB (visible light).

- False color imagery using short-wave infrared, near-infrared, and red for a .

- Creating a false color image and depicting the fire perimeters using boundaries to represent Palisades and Eaton fires.


### About the data

For this notebook, there are two major datasets:

1. Landsat 8 satellite

- The first data set, collected by Landsat 8 satellite, is used for its collection of bands (red, green, blue, near-infrared and short-wave infrared), which is from the Landsat Collection 2 Level-2 atmospherically corrected surface reflectance data.
- The data was retrieved from [Microsoft Planetary Computer data catalog](https://planetarycomputer.microsoft.com/dataset/landsat-c2-l2), though the data was already given by the instructor and clipped to the surrounding areas for the fire perimeters of both Palisades and Eaton, found in a Google Drive. Lastly, the landsat data is filed under the date it was acquired, February 23, 2025.

2. Palisades and Eaton Fire Perimeter

- The second dataset includes both the fire perimeters for Palisades and Eaton which were retrieved from [ArcGIS](https://hub.arcgis.com/maps/ad51845ea5fb4eb483bc2a7c38b2370c/about).
- For this analysis, we will be using the geometry for the boundary information needed for both fires.


### References

County of Los Angeles. (2025). *Palisades and Eaton dissolved fire perimeters* [Dataset]. ArcGIS Hub. https://hub.arcgis.com/maps/ad51845ea5fb4eb483bc2a7c38b2370c 

NASA. (2025, December 8). *Why is that forest red and that cloud blue? How to interpret a false-color satellite image*. https://science.nasa.gov/earth/earth-observatory/how-to-interpret-a-false-color-satellite-image/

U.S. Geological Survey. (2025). Landsat Collection 2 Level-2 [Dataset]. Microsoft Planetary Computer. https://planetarycomputer.microsoft.com/dataset/landsat-c2-l2 

In [ ]:
# Import libraries

import pandas as pd
import os
import numpy as np
import geopandas as gpd
import xarray as xr
import rioxarray
import matplotlib.pyplot as plt

## 2. Fire Perimeter Data Exploration

Explore the fire perimeter data and write a brief summary of the information you obtained from the preliminary exploration. Your summary should include, at least, the CRS of the data and whether this is projected or geographic.

In [ ]:
# Import data

fp = os.path.join(
    'data', 'Eaton_Perimeter_20250121',
    'Eaton_Perimeter_20250121.shp')
eaton = gpd.read_file(fp)

fp = os.path.join(
    'data', 'Palisades_Perimieter_20250121',
    'Palisades_Perimieter_20250121.shp')
palisades = gpd.read_file(fp)


### Explore CRS

In [ ]:
# Explore CRS for both Palisades and Eaton
print(f"Palisades' CRS: {palisades.crs}")
print(f"Eaton's CRS: {eaton.crs}")

In [ ]:
# Explore geographic or projected for Eaton 
print('Is Eaton geographic?: ', eaton.crs.is_geographic)
print('Is Eaton projected?: ', eaton.crs.is_projected)

In [ ]:
# Explore geographic or projected for Palisades 
print('Is Palisades geographic?: ', palisades.crs.is_geographic)
print('Is Palisades projected?: ', palisades.crs.is_projected)

### Exploration Palisades and Eaton data

In [ ]:
# Explore Palisades df
palisades.head()

In [ ]:
# Explore Eaton's df
eaton.head()

In [ ]:
# Plot Eaton
eaton.plot()

In [ ]:
# Plot Palisades
palisades.plot()

### Exploration summary

The CRS for both Palisades and Eaton is EPSG:4269 and for both they are geographic. They will need to be reprojected in order to match the CRS of Landsat to create the maps. Further exploration was done using `.head()` which contains the location, the geometry, and the year, as well as more. Plotting for the two areas were done to visualize for later perimeter usge for the true and false color images.

## 3. NetCDF data import and exploration

In [ ]:
# Load Landsat 8 data

landsat = xr.open_dataset('data/landsat8-2025-02-23-palisades-eaton.nc')
landsat

### Summary of Landsat exploration

Landsat has the following variables: red, green, blue, nir08, swir22, and spatial_ref (has the CRS info).

There are five spectral bands:
1. Visible light: red, green, and blue.
2. Near-infrared, nir08, discerns land-water boundaries that are not obvious in visible light and shows vegetation health due to the strong reflection which is the key point here.
3. Swir22 is a short-wave infrared band that helps scientists estimate water presence in plants and soil which is helpful for mapping fire damage as more water means darker images.

 The dimensions are: (y:1418, x: 2742) 

 The coordinates show y, x, (in meters) and time (February 23, 2025).

### Further readings

Readings for color images and Landsat imagery:

[NASA Earth Observatory - Why is that Forest Red and that Cloud Blue? How to Interpret a False-Color Satellite Image](https://science.nasa.gov/earth/earth-observatory/how-to-interpret-a-false-color-satellite-image/)

[What are the band designations for the Landsat satellites?](https://www.usgs.gov/faqs/what-are-band-designations-landsat-satellites)

[Common Landsat Band Combinations](https://www.usgs.gov/media/images/common-landsat-band-combinations)

## 4. Restoring geospatial information

In [ ]:
# Find CRS for landsat
print(landsat.rio.crs)

### Is this a geospatial object?

No.

After printing `landsat.rio.crs`, there was a `None` output, which means Python does not see this as a geospatial object. Recover geospatial information by using `rio.write_crs()` after retrieving from `spatial_ref.crs_wkt`.

In [ ]:
# Print CRS of landsat from variable spatial_ref
print(landsat.spatial_ref.crs_wkt)
landsat_crs = landsat.spatial_ref.crs_wkt

In [ ]:
# Recover CRS
landsat = landsat.rio.write_crs(landsat_crs)

In [ ]:
# Check again for CRS
print(landsat.rio.crs)

## 5. True color image
Without creating any new variables:

- select the red, green, and blue variables (in that order) of the xarray.Dataset holding the Landsat data,
- convert it to an xarray.DataArray using the to_dataarray() method, and then
- use .plot.imshow() to create an RGB image with the data. There will be a warning; that’s ok.


In [ ]:
# Select RGB and add no adjustments
landsat[['red', 'green', 'blue']].to_dataarray().plot.imshow()

### Adjust the scale used for plotting the bands to get a true color image.

In [ ]:
# Select RGB to create image and adjust scale
fig, ax = plt.subplots(figsize=(8, 6))
ax.axis('off')

landsat[['red', 'green', 'blue']].to_array().plot.imshow(ax=ax, robust=True)

ax.set_title('True Color Image Palisades and Eaton Fire Scars', 
             fontsize=15, 
             weight='bold', 
             pad=15
             )
plt.show()

### To resolve the other warning, identify which bands have nan values.

In [ ]:
# Identify NaN values
bands = ['red', 'green', 'blue', 'nir08', 'swir22']

for band in bands:
    count_nan = np.isnan(landsat[band]).sum().values
    print(f"{band} has {count_nan} NaN")

### Use the .fillna() method for xarray.Datasets to replace any nan values in the Landsat data with zero.

In [ ]:
# Replace NaN values in Landsat with 0
landsat = landsat.fillna(0)

In [ ]:
# Recreate image with no warnings
fig, ax = plt.subplots(figsize=(8, 6))
ax.axis('off')

landsat[['red', 'green', 'blue']].to_dataarray().plot.imshow(ax=ax, robust=True)

ax.set_title('True Color Image Palisades and Eaton Fire Scars', 
             fontsize=15, 
             weight='bold', 
             pad=15
             )
plt.show()

### Summary for true color images

A few adjustments were needed before the final true color image of the burned areas for Palisades and Eaton was created due to the two warnings which are listed here:

1. **Clipping Warning:** "Clipping input data to the valid range for imshow with RGB data ([0..1] for floats or [0..255] for integers)." This was addressed by using `robust=True`, which adjusts the scale used for plotting the bands to get a visible image as clouds were most likely the culprit for the bright pixels.


2. **"RuntimeWarning:** "invalid value encountered in cast xx = (xx * 255).astype(np.uint8)" After accounting for `NaN` resolved the warning since those values were missing and incapable of being converted into integers when mapped, which came from the bands, `green` and `blue`.

3. Comparing the first and last output in this section, the first had two warnings and showed completely white as displayed as it was outside the imshow's range. This was adjusted by resolving the warnings, as previously described, which allowed for a true color image to be shown as the last output.



## 6. False color image

In [ ]:
# Create false color image by plotting swir22, nir08, and red in order
fig, ax = plt.subplots(figsize=(8, 6))
ax.axis('off')

landsat[['swir22', 'nir08', 'red']].to_dataarray().plot.imshow(
    ax=ax, robust=True)

ax.set_title('False Color Image Palisades and Eaton Fire Scars', 
             fontsize=15, 
             weight='bold', 
             pad=15
             )
plt.show()

## 7. Map

In [ ]:
# Reproject perimeters to match landsat CRS
eaton = eaton.to_crs(landsat.rio.crs)
palisades = palisades.to_crs(landsat.rio.crs)

# Check CRS match with assert
assert eaton.crs == landsat.rio.crs
assert palisades.crs == landsat.rio.crs

In [ ]:
# False Color Map
fig, ax = plt.subplots(figsize=(8, 6))

# Plot false color image
landsat[['swir22', 'nir08', 'red']].to_dataarray().plot.imshow(ax=ax, robust=True)

# Plot fire perimeter polygons on top of the image
eaton.boundary.plot(ax=ax,
                    color="navy",
                      linewidth=1, label="Eaton Fire Perimeter")
palisades.boundary.plot(ax=ax,
                         color="gold",
                           linewidth=1, label="Palisades Fire Perimeter")

# Add fire names as text annotations
for fire, name in [(eaton, 'Eaton'), (palisades, 'Palisades')]:
    center = fire.geometry.centroid.iloc[0]
    ax.text(center.x, center.y + 5500, name, fontsize=8,
             ha='center', weight='bold', bbox=dict(
                 boxstyle='round,pad=0.3',
                facecolor='white',
                edgecolor='black', alpha=0.8)
)

# Add labels
ax.set_title('Eaton and Palisades Fire Scars via False Color Image', 
             fontsize=15, 
             weight='bold', 
             pad=15
             )
plt.legend()
ax.set_xlabel('X coordinate (meters)')
ax.set_ylabel('Y coordinate (meters)')
fig.text(
    0.99,
    0.01,
    "Data Source: Landsat (USGS, via Microsoft Planetary Computer); "
         "Fire perimeters (County of Los Angeles, via ArcGIS Hub) 2025.",
    ha="right",
    va="bottom",
    fontsize=9,
    color="gray",
    style="italic",
)

ax.ticklabel_format(style='plain', useOffset=False)

plt.show()

### Eaton and Palisades Fire Scars via False Color Image figure description

This false color image shows the affected areas, in reddish-brown, for both areas from the fires that started in early January in 2025. The gold lines represent the fire perimeter for the Palisades, which fit the fire scars along its border. Eaton fire scars are contained with an overly large perimeter represented by the navy lines.

This image was created by using data collected by Landsat, weeks later after the fires, on February 23, 2025. Short-wave infrared (swir22) and near-infrared (nir08) as well as red bands were used in that exact order to differentiate between the burn areas (red due to SWIR) and vegetation (green due to NIR). The short-wave infrared shows bodies of water as darkest since it absorbs it, which may not be important, but can explain why the water is displayed as black. The near-infrared shows the vegetation health as green since it reflects NIR strongly, which makes using false color crucial for fire depictions. Thus, the newly burned land reflects SWIR bands strongly and makes burned areas easily visible.
